# VWAP Events 02 — touches, crosses and time to return (descriptive)

BTC and ETH, Binance USD-M perpetual 1m. Session VWAP anchored 08:00 UTC; 4H VWAP anchored 00/04/08/12/16/20 UTC (definitions as VWAP Events 01).

- **Touch:** bar range contains the VWAP known at the bar's open. Consecutive touch bars = one *episode*.
- **Cross:** close switches side of VWAP (within the same anchor window).
- **After a cross:** minutes until the next **touch** (price back at VWAP from the new side), until a **cross back**, and until a touch **after price first detached** (removes the trivial "still sitting on it" case). Crosses whose window resets first are censored; Kaplan-Meier medians account for that.
- **Placebo:** the same statistics for VWAPs anchored at unused times (session at 02/14/20 UTC; 4H offset by 1/2/3 h).

Descriptive only: no gates, no trading claim.

In [1]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
pd.set_option('display.width',250);pd.set_option('display.max_columns',40)
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import vwap_events as ve, vwap_touch_stats as vt
importlib.reload(vt)
SYNTHETIC=os.getenv('VE02_SYNTHETIC','0')=='1'
ASSETS=['BTC','ETH']
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='ve02-');OUT=Path(tmp.name)/'02'
else:
    OUT=ROOT/'notebooks'/'VWAP_Events'/'02'
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_vwap_touch_stats.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
print(chk.stdout[-400:]);assert chk.returncode==0,'tests failed'
print(vt.VERSION)

.....                                                                    [100%]
5 passed in 1.71s

VE02-0.1.0


In [2]:
CNT,SUM,EV=[],[],[]
for i,a in enumerate(ASSETS):
    t0=time.time()
    df=ve.synthetic_klines(n_days=120,seed=5+i) if SYNTHETIC else pd.read_parquet(BIN/f'{a}USDT_um_1m.parquet',columns=['open_time','high','low','close','volume','quote_volume'])
    c,s,e=vt.run_asset(df,a);CNT.append(c);SUM.append(s);EV.append(e)
    print(a,f'{df.open_time.min()} -> {df.open_time.max()}',f'{time.time()-t0:.0f}s')
CNT=pd.concat(CNT,ignore_index=True);SUM=pd.concat(SUM,ignore_index=True);EV=pd.concat(EV,ignore_index=True)
CNT.to_csv(OUT/'02_counts.csv',index=False);SUM.to_csv(OUT/'02_return_times.csv',index=False)

BTC 2024-10-01 00:00:00+00:00 -> 2026-10-11 23:59:00+00:00 16s
ETH 2024-10-01 00:00:00+00:00 -> 2026-10-11 23:59:00+00:00 17s


## 1. How often: touches and crosses

In [3]:
show=CNT[CNT.kind=='real'][['asset','anchor','days','windows','touch_bars','touch_episodes','touch_episodes_per_window','crosses','crosses_up','crosses_down','crosses_per_window','crosses_per_day']]
display(show.round(2))
print('Real vs placebo anchors (touch episodes and crosses per window):')
display(CNT.pivot_table(index=['asset','anchor'],columns='kind',values=['touch_episodes_per_window','crosses_per_window']).round(2))

,asset,anchor,days,windows,touch_bars,touch_episodes,touch_episodes_per_window,crosses,crosses_up,crosses_down,crosses_per_window,crosses_per_day
0,BTC,session,741.0,742,47330,25757,34.71,30594,15302,15292,41.23,41.29
4,BTC,h4,741.0,4446,107741,59462,13.37,69562,34854,34708,15.65,93.88
8,ETH,session,741.0,742,53830,27064,36.47,31535,15782,15753,42.50,42.56
12,ETH,h4,741.0,4446,122248,62002,13.95,72100,36144,35956,16.22,97.30


Real vs placebo anchors (touch episodes and crosses per window):


crosses_per_window        touch_episodes_per_window       
kind                     placebo   real                   placebo   real
asset anchor                                                            
BTC   h4                   15.83  15.65                     13.60  13.37
      session              39.96  41.23                     34.39  34.71
ETH   h4                   16.56  16.22                     14.28  13.95
      session              41.09  42.50                     35.72  36.47

## 2. After a cross: time until price comes back

In [4]:
s=SUM[(SUM.kind=='real')]
cols=['asset','anchor','measure','direction','n','share_returned_before_reset','median_min_if_returned','mean_min_if_returned','km_median_min','share_within_1m','share_within_5m','share_within_15m','share_within_60m']
for m in ('min_to_touch','min_to_return_after_detach' if False else 'min_return_after_detach','min_to_cross_back'):
    print(m);display(s[s.measure==m][cols].round(3))

min_to_touch


,asset,anchor,measure,direction,n,share_returned_before_reset,median_min_if_returned,mean_min_if_returned,km_median_min,share_within_1m,share_within_5m,share_within_15m,share_within_60m
0,BTC,session,min_to_touch,all,30594,0.984,2.0,15.675,2.0,0.443,0.709,0.835,0.929
1,BTC,session,min_to_touch,up,15302,0.983,2.0,16.996,2.0,0.444,0.704,0.829,0.923
2,BTC,session,min_to_touch,down,15292,0.986,2.0,14.356,2.0,0.442,0.713,0.841,0.936
36,BTC,h4,min_to_touch,all,69562,0.958,2.0,7.590,2.0,0.437,0.711,0.844,0.936
37,BTC,h4,min_to_touch,up,34854,0.956,2.0,7.632,2.0,0.438,0.712,0.843,0.933
38,BTC,h4,min_to_touch,down,34708,0.960,2.0,7.548,2.0,0.437,0.711,0.845,0.939
72,ETH,session,min_to_touch,all,31535,0.986,2.0,13.733,2.0,0.491,0.738,0.854,0.938
73,ETH,session,min_to_touch,up,15782,0.985,2.0,13.945,2.0,0.489,0.737,0.853,0.935
74,ETH,session,min_to_touch,down,15753,0.987,1.0,13.521,2.0,0.494,0.740,0.854,0.942
108,ETH,h4,min_to_touch,all,72100,0.963,1.0,6.720,2.0,0.484,0.742,0.862,0.945


min_return_after_detach


,asset,anchor,measure,direction,n,share_returned_before_reset,median_min_if_returned,mean_min_if_returned,km_median_min,share_within_1m,share_within_5m,share_within_15m,share_within_60m
6,BTC,session,min_return_after_detach,all,30594,0.972,5.0,26.556,5.0,0.224,0.526,0.719,0.876
7,BTC,session,min_return_after_detach,up,15302,0.970,5.0,27.636,5.0,0.223,0.522,0.714,0.869
8,BTC,session,min_return_after_detach,down,15292,0.974,5.0,25.479,5.0,0.226,0.529,0.724,0.882
42,BTC,h4,min_return_after_detach,all,69562,0.926,4.0,11.979,5.0,0.223,0.533,0.737,0.889
43,BTC,h4,min_return_after_detach,up,34854,0.924,4.0,12.025,5.0,0.226,0.532,0.735,0.886
44,BTC,h4,min_return_after_detach,down,34708,0.928,4.0,11.934,5.0,0.221,0.533,0.739,0.891
78,ETH,session,min_return_after_detach,all,31535,0.973,4.0,26.036,5.0,0.228,0.535,0.730,0.881
79,ETH,session,min_return_after_detach,up,15782,0.973,4.0,26.202,5.0,0.227,0.535,0.730,0.878
80,ETH,session,min_return_after_detach,down,15753,0.974,4.0,25.870,5.0,0.229,0.534,0.729,0.884
114,ETH,h4,min_return_after_detach,all,72100,0.927,4.0,11.253,4.0,0.232,0.546,0.747,0.896


min_to_cross_back


,asset,anchor,measure,direction,n,share_returned_before_reset,median_min_if_returned,mean_min_if_returned,km_median_min,share_within_1m,share_within_5m,share_within_15m,share_within_60m
3,BTC,session,min_to_cross_back,all,30594,0.976,3.0,23.799,4.0,0.290,0.592,0.759,0.891
4,BTC,session,min_to_cross_back,up,15302,0.974,3.0,24.851,4.0,0.291,0.590,0.755,0.884
5,BTC,session,min_to_cross_back,down,15292,0.978,3.0,22.750,3.0,0.289,0.595,0.763,0.897
39,BTC,h4,min_to_cross_back,all,69562,0.936,3.0,10.810,4.0,0.285,0.592,0.768,0.902
40,BTC,h4,min_to_cross_back,up,34854,0.933,3.0,10.737,3.0,0.286,0.595,0.768,0.899
41,BTC,h4,min_to_cross_back,down,34708,0.939,3.0,10.882,4.0,0.285,0.590,0.768,0.905
75,ETH,session,min_to_cross_back,all,31535,0.976,3.0,22.973,4.0,0.293,0.594,0.763,0.895
76,ETH,session,min_to_cross_back,up,15782,0.974,3.0,23.203,4.0,0.292,0.591,0.762,0.892
77,ETH,session,min_to_cross_back,down,15753,0.979,3.0,22.745,3.0,0.294,0.596,0.765,0.898
111,ETH,h4,min_to_cross_back,all,72100,0.938,3.0,10.516,3.0,0.291,0.601,0.775,0.906


## 3. Real vs placebo anchors

In [5]:
rvp=vt.real_vs_placebo(SUM);rvp.to_csv(OUT/'02_real_vs_placebo.csv',index=False);display(rvp.round(3))

,asset,anchor,measure,share_returned_before_reset,median_min_if_returned,km_median_min,share_within_5m,share_returned_before_reset_placebo,median_min_if_returned_placebo,km_median_min_placebo,share_within_5m_placebo
0,BTC,session,min_to_touch,0.984,2.0,2.0,0.709,0.984,2.000,2.000,0.707
1,BTC,session,min_to_cross_back,0.976,3.0,4.0,0.592,0.975,3.333,4.000,0.587
2,BTC,session,min_return_after_detach,0.972,5.0,5.0,0.526,0.972,5.000,5.000,0.523
3,BTC,h4,min_to_touch,0.958,2.0,2.0,0.711,0.958,2.000,2.000,0.711
4,BTC,h4,min_to_cross_back,0.936,3.0,4.0,0.592,0.937,3.000,4.000,0.592
5,BTC,h4,min_return_after_detach,0.926,4.0,5.0,0.533,0.925,4.000,5.000,0.533
6,ETH,session,min_to_touch,0.986,2.0,2.0,0.738,0.986,1.667,2.000,0.737
7,ETH,session,min_to_cross_back,0.976,3.0,4.0,0.594,0.976,3.000,3.333,0.592
8,ETH,session,min_return_after_detach,0.973,4.0,5.0,0.535,0.972,4.000,5.000,0.531
9,ETH,h4,min_to_touch,0.963,1.0,2.0,0.742,0.964,1.333,2.000,0.743


## 4. Distribution of return times after detaching (real anchors, minutes)

In [6]:
q=EV[EV.kind=='real'].groupby(['asset','anchor'])['min_return_after_detach'].describe(percentiles=[.1,.25,.5,.75,.9]).round(1)
display(q);q.to_csv(OUT/'02_return_after_detach_quantiles.csv')
print('time of day of crosses (UTC hour, share):')
h=EV[EV.kind=='real'].assign(hour=lambda d:pd.to_datetime(d.t).dt.hour).groupby(['asset','anchor']).hour.value_counts(normalize=True).unstack().round(3)
display(h);h.to_csv(OUT/'02_cross_hour_share.csv')
(OUT/'02_manifest.json').write_text(json.dumps({'version':vt.VERSION,'assets':ASSETS,'synthetic':SYNTHETIC},indent=1))

count  mean   std  min  10%  25%  50%   75%   90%     max
asset anchor                                                              
BTC   h4       64389.0  12.0  20.8  1.0  1.0  2.0  4.0  12.0  32.0   235.0
      session  29742.0  26.6  73.4  1.0  1.0  2.0  5.0  16.8  60.0  1261.0
ETH   h4       66862.0  11.3  19.5  1.0  1.0  1.0  4.0  12.0  30.0   226.0
      session  30698.0  26.0  75.6  1.0  1.0  2.0  4.0  16.0  58.0  1319.0

time of day of crosses (UTC hour, share):


hour              0      1      2      3      4      5      6      7      8      9      10     11     12     13     14     15     16     17     18     19     20     21     22     23
asset anchor                                                                                                                                                                         
BTC   h4       0.073  0.039  0.027  0.022  0.067  0.039  0.030  0.026  0.072  0.034  0.029  0.026  0.075  0.046  0.036  0.027  0.073  0.038  0.029  0.025  0.070  0.040  0.030  0.025
      session  0.026  0.030  0.024  0.023  0.019  0.017  0.019  0.019  0.163  0.078  0.065  0.060  0.065  0.067  0.058  0.046  0.035  0.032  0.030  0.028  0.028  0.024  0.024  0.020
ETH   h4       0.076  0.039  0.027  0.022  0.070  0.037  0.030  0.026  0.072  0.038  0.031  0.026  0.077  0.048  0.036  0.026  0.071  0.036  0.029  0.023  0.068  0.040  0.030  0.024
      session  0.029  0.025  0.022  0.020  0.019  0.017  0.019  0.020  0.165  0.087  0.071  0.060  0.060  0.067  0.056  0.043  0.036  0.032  0.032  0.024  0.023  0.026  0.023  0.023

83